In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Olist-Pipeline") \
    .master("local[*]") \
    .getOrCreate()

In [2]:
hadoop_conf = spark._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://localhost:9000")
hadoop_conf.set("fs.s3a.access.key", "minioadmin")
hadoop_conf.set("fs.s3a.secret.key", "minioadmin")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

In [5]:
df_orders = spark.read.csv(
    "s3a://my-bucket/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

df_orders.show(5)
df_orders.printSchema()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [ ]:
df_orders = df_orders.dropDuplicates(["order_id"])


In [5]:
orders_df.show(5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:3

In [6]:
print("Rows:", orders_df.count())

Rows: 99441


In [7]:
from pyspark.sql.functions import col

total = orders_df.count()
unique = orders_df.dropDuplicates(["order_id"]).count()

print("Total:", total)
print("Unique:", unique)
print("Duplicates:", total - unique)

Total: 99441
Unique: 99441
Duplicates: 0


In [8]:
orders_df = orders_df.dropDuplicates(["order_id"])

In [9]:
from pyspark.sql.functions import col, count, when

orders_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_df.columns
]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [10]:
orders_df.show(5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|00018f77f2f0320c557190d7a144bdd3|f6dd3ec061db4e3987629fe6b26e5cce|delivered   |2017-04-26 10:53:06     |2017-04-26 11:05:13|2017-05-04 14:35:00         |2017-05-12 16:04:24          |2017-05-15 00:00:00          |
|00042b26cf59d7ce69dfabb4e55b4fd9|58dbd0b2d70206bf40e62cd34e84d795|delivered   |2017-02-04 13:57:51     |2017-02-04 14:10:13|2017-02-16 09:4

In [7]:
df_orders.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/olist_db") \
    .option("dbtable", "ods_orders") \
    .option("user", "postgres") \
    .option("password", "rootroot") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [12]:
from pyspark.sql.functions import col, count, when

orders_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_df.columns
]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [2]:
import psycopg2

conn = psycopg2.connect(
    dbname="olist_db",
    user="postgres",
    password="rootroot",
    host="localhost",
    port="5432"
)

print("Connected successfully!")

Connected successfully!


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Olist-Pipeline") \
    .master("local[*]") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "org.postgresql:postgresql:42.7.3") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

In [4]:
spark.sparkContext._jsc.sc().listJars()

JavaObject id=o42

In [ ]:
def load_clean_to_postgres(file_name, table_name, key_column):

    df = spark.read.csv(
        f"s3a://my-bucket/{file_name}",
        header=True,
        inferSchema=True,
        quote='"',
        escape='"',
        multiLine=True
    )

    print("\n" + "="*50)
    print(f"Processing: {file_name}")
    print("Raw Schema:")
    df.printSchema()
    print(" Raw Rows:", df.count())

    df = df.dropDuplicates([key_column])

    print("\n Cleaned Schema:")
    df.printSchema()
    print("Clean Rows:", df.count())

    # 5) Write to PostgreSQL (ODS)
    jdbc_url = "jdbc:postgresql://127.0.0.1:5432/olist_db"

    df.write \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", table_name) \
        .option("user", "postgres") \
        .option("password", "rootroot") \
        .option("driver", "org.postgresql.Driver") \
        .option("batchsize", "5000") \
        .option("numPartitions", "1") \
        .option("isolationLevel", "NONE") \
        .mode("append") \
        .save()

    print(f" Done: {table_name}")
    print("="*50 + "\n")

In [34]:
load_clean_to_postgres(
    "olist_orders_dataset.csv",
    "ods_orders",
    "order_id"
)


Processing: olist_orders_dataset.csv
📊 Raw Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

📏 Raw Rows: 99441

🧹 Cleaned Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

📏 Clean Rows: 99441
✅ Done: ods_orders



In [9]:
load_clean_to_postgres(
    "olist_customers_dataset.csv",
    "ods_customers",
    "customer_id"
)

Processing: olist_customers_dataset.csv
Rows: 99441
Done: ods_customers


In [18]:
load_clean_to_postgres(
    "olist_order_payments_dataset.csv",
    "ods_order_payments",
    "order_id"
)


Processing: olist_order_payments_dataset.csv
📊 Raw Schema:
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)

📏 Raw Rows: 103886

🧹 Cleaned Schema:
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)

📏 Clean Rows: 99440
✅ Done: ods_order_payments



In [23]:
load_clean_to_postgres("olist_order_items_dataset.csv", 
                       "ods_order_items",
                       "order_id")


Processing: olist_order_items_dataset.csv
📊 Raw Schema:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

📏 Raw Rows: 112650

🧹 Cleaned Schema:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

📏 Clean Rows: 98666
✅ Done: ods_order_items



In [21]:
import psycopg2

conn = psycopg2.connect(
    dbname="olist_db",
    user="postgres",
    password="rootroot",
    host="127.0.0.1",
    port="5432"
)

print("CONNECTED SUCCESS")
conn.close()

CONNECTED SUCCESS


In [33]:
load_clean_to_postgres(
    "olist_order_reviews_dataset.csv",
    "ods_order_reviews",
    "review_id"
)


Processing: olist_order_reviews_dataset.csv
📊 Raw Schema:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

📏 Raw Rows: 99224

🧹 Cleaned Schema:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

📏 Clean Rows: 98410
✅ Done: ods_order_reviews



In [35]:
load_clean_to_postgres(
    "olist_products_dataset.csv",
    "ods_products",
    "product_id"
)


Processing: olist_products_dataset.csv
📊 Raw Schema:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

📏 Raw Rows: 32951

🧹 Cleaned Schema:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nulla

In [26]:
load_clean_to_postgres(
    "olist_sellers_dataset.csv",
    "ods_sellers",
    "seller_id"
)


Processing: olist_sellers_dataset.csv
📊 Raw Schema:
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

📏 Raw Rows: 3095

🧹 Cleaned Schema:
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

📏 Clean Rows: 3095
✅ Done: ods_sellers



In [27]:
load_clean_to_postgres(
    "olist_geolocation_dataset.csv",
    "ods_geolocation",
    "geolocation_zip_code_prefix"
)


Processing: olist_geolocation_dataset.csv
📊 Raw Schema:
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

📏 Raw Rows: 1000163

🧹 Cleaned Schema:
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

📏 Clean Rows: 19015
✅ Done: ods_geolocation



In [28]:
load_clean_to_postgres(
    "product_category_name_translation.csv",
    "ods_product_category_translation",
    "product_category_name"
)


Processing: product_category_name_translation.csv
📊 Raw Schema:
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)

📏 Raw Rows: 71

🧹 Cleaned Schema:
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)

📏 Clean Rows: 71
✅ Done: ods_product_category_translation

